# 🎯 CS2 Bot Training - Обучение на топовых матчах
## Сжатые демки с HLTV для экономии места

**Что делает:**
1. Скачивает демки топовых команд с HLTV на твоём ПК
2. Сжимает их в ZIP архив (экономия ~30-50% места)
3. В Colab автоматически распаковывает и парсит
4. Обучает AI на про-игре

**Требования:**
- GPU Runtime (Runtime → Change runtime type → T4 GPU)
- Сжатый архив демок на Google Drive

**Преимущества:**
- Меньше места на Drive (~10-15 GB вместо 20 GB)
- Быстрее загружается в Colab
- Демки с HLTV (без блокировок)

## 🔧 Шаг 1: Проверка GPU

In [ ]:
# Проверка GPU (должен быть Tesla T4 или другой)
!nvidia-smi

## 📦 Шаг 2: Установка библиотек

In [ ]:
# Установка всех необходимых библиотек (~2 минуты)
!pip install -q requests awpy pandas pyarrow tqdm torch

print("✅ Библиотеки установлены!")

## 📥 Шаг 3: Клонирование репозитория

In [ ]:
# Удаляем старые копии если есть
!rm -rf /content/cs2-bot-training

# Переходим в корень
%cd /content

# Клонируем репозиторий
!git clone https://github.com/nem1k9/cs2-bot-training.git

# Переходим в папку scripts (там все .py файлы)
%cd cs2-bot-training/scripts

# Проверяем что файлы есть
print("\n✅ Файлы проекта:")
!ls -la

## 🎮 Шаг 4: Загрузка демок

**Инструкция:**
1. Скачай демки на своём ПК: `python download_and_compress.py`
2. Загрузи архив `cs2_demos_compressed.zip` на Google Drive
3. Запусти ячейку ниже - она автоматически распакует архив

### 📥 Подключение Drive и копирование демок

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive подключен!")

In [ ]:
import os
import zipfile
from tqdm import tqdm

# ========== НАСТРОЙКИ ==========
DRIVE_ZIP_PATH = "/content/drive/MyDrive/cs2_demos_compressed.zip"  # ← Путь к архиву на Drive
PLAYER = "donk"  # ← Имя игрока для парсинга (опционально)
# ===============================

print(f"📦 Распаковываем демки из архива...\n")

# Создаём папку для демок
!mkdir -p ./demos

# Проверяем что архив существует
if not os.path.exists(DRIVE_ZIP_PATH):
    print(f"❌ ОШИБКА: Архив не найден!")
    print(f"📁 Ожидаемый путь: {DRIVE_ZIP_PATH}")
    print(f"\n💡 РЕШЕНИЕ:")
    print(f"   1. Запусти на своём ПК: python download_and_compress.py")
    print(f"   2. Загрузи файл cs2_demos_compressed.zip на Google Drive")
    print(f"   3. Убедись что путь правильный")
else:
    # Получаем размер архива
    zip_size_gb = os.path.getsize(DRIVE_ZIP_PATH) / 1e9
    print(f"✅ Архив найден: {zip_size_gb:.2f} GB")
    
    # Распаковываем
    print(f"📦 Распаковываем...\n")
    
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
        # Получаем список файлов
        file_list = [f for f in zip_ref.namelist() if f.endswith('.dem')]
        
        print(f"📊 Файлов в архиве: {len(file_list)}")
        
        # Распаковываем с прогресс-баром
        for file in tqdm(file_list, desc="Распаковка"):
            zip_ref.extract(file, './demos/')
    
    print(f"\n✅ Распаковано {len(file_list)} демок!")
    
    # Проверяем результат
    print("\n📊 Демки в ./demos/:")
    !ls -lh ./demos/*.dem | wc -l
    !ls -lh ./demos/*.dem | head -5
    
    print(f"\n✅ Демки готовы к парсингу!")

## 🔍 Шаг 5: Парсинг демок

**Что отслеживается:**
- Движение (позиция, скорость, присед)
- Прицеливание (yaw, pitch, скоп)
- Стрельба (выстрелы, перезарядка)
- Гранаты (флешки, смоки, HE, молотовы)
- Тактика (позиционирование, дистанция до врагов)

**Примечание:** Если указан PLAYER, парсятся только его действия. Если нет - все игроки.

In [ ]:
from parse_demos import build_dataset

print(f"🔄 Парсим демки...")
if PLAYER:
    print(f"🎯 Фокус на игроке: {PLAYER}")
else:
    print(f"🎯 Парсим всех игроков")
    
print(f"⏱️  Это займёт ~10-30 минут...\n")

# Парсим демки (с фильтром по игроку если указан)
dataset = build_dataset(
    demo_dir="./demos",
    out_path="./dataset.parquet",
    target_player=PLAYER if PLAYER else None
)

if PLAYER:
    print(f"\n✅ Датасет для {PLAYER} готов: {len(dataset):,} тиков")
else:
    print(f"\n✅ Датасет готов: {len(dataset):,} тиков")
    
print(f"📊 Размер датасета: {dataset.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 🚀 Шаг 6: Обучение модели

**Время обучения:** ~2-4 часа

**ВАЖНО:** Не закрывай вкладку браузера!

In [ ]:
from train import train

print(f"🚀 Начинаем обучение модели на стиле {PLAYER}...")
print(f"⏱️  Это займёт ~2-4 часа...\n")

train()

print(f"\n✅ Обучение завершено!")
print(f"🎯 Модель обучена копировать стиль игры {PLAYER}!")

## 💾 Шаг 7: Сохранение результатов на Google Drive

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Создаём папку для результатов
!mkdir -p /content/drive/MyDrive/cs2_bot_results

# Сохраняем модели и датасет
print("💾 Сохраняем результаты на Google Drive...\n")

!cp cs2bot_final.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ cs2bot_final.pt сохранён" || echo "⚠️  cs2bot_final.pt не найден"
!cp checkpoint.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ checkpoint.pt сохранён" || echo "⚠️  checkpoint.pt не найден"
!cp dataset.parquet /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ dataset.parquet сохранён" || echo "⚠️  dataset.parquet не найден"

print("\n✅ Результаты сохранены на Google Drive в папке cs2_bot_results!")
print("\n📁 Что сохранено:")
!ls -lh /content/drive/MyDrive/cs2_bot_results/

## 📥 Шаг 8: Скачать модель на компьютер (опционально)

In [ ]:
# Скачать модель прямо в браузер
from google.colab import files

print("📥 Скачиваем модель...")
files.download('cs2bot_final.pt')
files.download('checkpoint.pt')

print("✅ Модель скачана!")

## 🎯 Готово!

**Что получилось:**
- ✅ Скачаны демки топовых команд с HLTV
- ✅ Сжаты в ZIP архив (экономия места ~30-50%)
- ✅ Распакованы в Colab
- ✅ Распарсены все действия игроков
- ✅ Обучена модель на про-игре
- ✅ Результаты сохранены на Google Drive

**Модель умеет:**
- Двигаться как про-игроки
- Целиться и стрелять
- Использовать гранаты
- Принимать тактические решения

**Это AI обученный на топовых матчах!** 🔥